In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("adult_clean_normalized.csv")

# target: income_high (0/1)
df['income_high'] = (df['income'] == '>50K').astype(int)

# -------------------------
# SAMPLE 1: RANDOM SPLIT
# -------------------------
df_sample = df.sample(2000, random_state=42)
A_random = df_sample.iloc[:1000]
B_random = df_sample.iloc[1000:]

# metrics
CR_A_rand = A_random['income_high'].mean()
CR_B_rand = B_random['income_high'].mean()

# z-test
count = np.array([A_random['income_high'].sum(), B_random['income_high'].sum()])
nobs = np.array([len(A_random), len(B_random)])
stat_rand, p_rand = proportions_ztest(count, nobs)

# -------------------------
# SAMPLE 2: PREDEFINED EFFECT
# -------------------------
A_pre = df.sample(5000, random_state=1)

# створюємо варіант B з вищим CR
high_income = df[df['income_high'] == 1].sample(1500, random_state=2)
low_income = df[df['income_high'] == 0].sample(3500, random_state=3)
B_pre = pd.concat([high_income, low_income]).sample(frac=1)

CR_A_pre = A_pre['income_high'].mean()
CR_B_pre = B_pre['income_high'].mean()

count2 = np.array([A_pre['income_high'].sum(), B_pre['income_high'].sum()])
nobs2 = np.array([len(A_pre), len(B_pre)])
stat_pre, p_pre = proportions_ztest(count2, nobs2)

# Secondary metric (hours-per-week)
t_stat, p_t = ttest_ind(A_pre['hours-per-week'], B_pre['hours-per-week'], equal_var=False)

# -------------------------
# PRINT RESULTS
# -------------------------
print("=== RANDOM SAMPLE ===")
print("CR A =", CR_A_rand)
print("CR B =", CR_B_rand)
print("p-value =", p_rand)

print("\n=== PREDEFINED SAMPLE ===")
print("CR A =", CR_A_pre)
print("CR B =", CR_B_pre)
print("p-value =", p_pre)
print("Secondary metric (hours/week) p-value =", p_t)


=== RANDOM SAMPLE ===
CR A = 0.225
CR B = 0.231
p-value = 0.7491301524642896

=== PREDEFINED SAMPLE ===
CR A = 0.244
CR B = 0.3
p-value = 3.128409180002089e-10
Secondary metric (hours/week) p-value = 0.012196192015006845
